# Bedrock Cost Optimization

Build a Knowledge Base from Amazon SEC filings, then apply cost optimization techniques.

## What you will do
1. Download Amazon 10-K filings from SEC EDGAR
2. Create an S3 bucket and upload the documents
3. Create a Bedrock Knowledge Base (S3 Vectors + Titan embeddings)
4. Sync and query the KB in 3 modes (Prompt Router, AIP, Prompt Caching)
5. Apply 5 cost optimization techniques

> **Estimated time:** 45 minutes
> **Cost:** ~$1-2 for the full workshop

---
## Step 1: Setup

In [ ]:
import boto3
import json
import time
import os
import urllib.request
from IPython.display import Markdown, display

sts = boto3.client('sts')
ACCOUNT_ID = sts.get_caller_identity()['Account']
REGION = boto3.session.Session().region_name or 'us-east-1'

print(f'Account: {ACCOUNT_ID}')
print(f'Region:  {REGION}')

# Clients
s3 = boto3.client('s3', region_name=REGION)
bedrock_agent = boto3.client('bedrock-agent', region_name=REGION)
bedrock_runtime = boto3.client('bedrock-runtime', region_name=REGION)
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime', region_name=REGION)
iam = boto3.client('iam')

# Config
BUCKET_NAME = f'bedrock-kb-workshop-{ACCOUNT_ID}-{REGION}'
KB_NAME = f'kb-{ACCOUNT_ID}-{time.strftime("%Y%m%d%H%M")}'
MODEL_ID = 'us.amazon.nova-pro-v1:0'
LITE_MODEL = 'us.amazon.nova-lite-v1:0'
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'

PRICING = {
    'us.amazon.nova-lite-v1:0': {'input': 0.00006, 'output': 0.00024},
    'us.amazon.nova-pro-v1:0': {'input': 0.0008, 'output': 0.0032},
    'anthropic.claude-sonnet-4-6': {'input': 0.003, 'output': 0.015},
}

def estimate_cost(model, input_tokens, output_tokens):
    p = PRICING.get(model, {'input': 0.0008, 'output': 0.0032})
    return (input_tokens/1000 * p['input']) + (output_tokens/1000 * p['output'])

# Save config for notebook 2
config = {'ACCOUNT_ID': ACCOUNT_ID, 'REGION': REGION, 'BUCKET_NAME': BUCKET_NAME, 'KB_NAME': KB_NAME, 'MODEL_ID': MODEL_ID, 'LITE_MODEL': LITE_MODEL, 'EMBEDDING_MODEL': EMBEDDING_MODEL}
with open('/tmp/certagent_config.json', 'w') as f:
    json.dump(config, f)
print('Setup complete (config saved to /tmp/certagent_config.json)')

---
## Step 2: Download Amazon SEC Filings

We download Amazon's recent 10-K annual reports from SEC EDGAR (public data).

In [ ]:
os.makedirs('/tmp/sec-filings', exist_ok=True)

# Amazon SEC filings - 10-K annual reports (public URLs)
filings = {
    'amazon-10k-2023.htm': 'https://www.sec.gov/Archives/edgar/data/1018724/000101872424000008/amzn-20231231.htm',
    'amazon-10k-2022.htm': 'https://www.sec.gov/Archives/edgar/data/1018724/000101872423000004/amzn-20221231.htm',
}

downloaded = []
for filename, url in filings.items():
    filepath = f'/tmp/sec-filings/{filename}'
    if not os.path.exists(filepath):
        print(f'Downloading {filename}...')
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Workshop/1.0 (workshop@example.com)'})
            with urllib.request.urlopen(req, timeout=30) as resp:
                with open(filepath, 'wb') as f:
                    f.write(resp.read())
            downloaded.append(filename)
            print(f'  Downloaded: {os.path.getsize(filepath) / 1024:.0f} KB')
        except Exception as e:
            print(f'  Error downloading {filename}: {e}')
            # Create a sample document as fallback
            with open(filepath, 'w') as f:
                f.write(f'<html><body><h1>Amazon.com Inc - Annual Report</h1><p>This is a sample document for the workshop. Amazon reported net sales of $574.8 billion in 2023, an increase of 12% year-over-year. AWS segment revenue was $90.8 billion. Operating income was $36.9 billion. The company employed approximately 1.5 million people worldwide.</p></body></html>')
            downloaded.append(filename)
            print(f'  Created sample: {filename}')
    else:
        downloaded.append(filename)
        print(f'  Already exists: {filename}')

print(f'\nFiles ready: {len(downloaded)}')

---
## Step 3: Create S3 Bucket and Upload Documents

In [ ]:
# Create bucket
try:
    if REGION == 'us-east-1':
        s3.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3.create_bucket(Bucket=BUCKET_NAME, CreateBucketConfiguration={'LocationConstraint': REGION})
    print(f'Created bucket: {BUCKET_NAME}')
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f'Bucket already exists: {BUCKET_NAME}')

# Upload documents
for filename in downloaded:
    filepath = f'/tmp/sec-filings/{filename}'
    key = f'documents/{filename}'
    s3.upload_file(filepath, BUCKET_NAME, key)
    print(f'  Uploaded: s3://{BUCKET_NAME}/{key}')

print(f'\nAll documents uploaded to s3://{BUCKET_NAME}/documents/')

---
## Step 4: Create IAM Role for Knowledge Base

In [ ]:
# Create KB execution role
KB_ROLE_NAME = f'BedrockKBRole-workshop'

trust_policy = {'Version': '2012-10-17', 'Statement': [{'Effect': 'Allow', 'Principal': {'Service': 'bedrock.amazonaws.com'}, 'Action': 'sts:AssumeRole'}]}

try:
    role = iam.create_role(RoleName=KB_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))
    KB_ROLE_ARN = role['Role']['Arn']
    print(f'Created role: {KB_ROLE_ARN}')
except iam.exceptions.EntityAlreadyExistsException:
    KB_ROLE_ARN = iam.get_role(RoleName=KB_ROLE_NAME)['Role']['Arn']
    print(f'Role exists: {KB_ROLE_ARN}')

# Attach permissions for S3, Bedrock, and S3 Vectors
kb_policy = {'Version': '2012-10-17', 'Statement': [
    {'Effect': 'Allow', 'Action': ['s3:GetObject', 's3:ListBucket'], 'Resource': [f'arn:aws:s3:::{BUCKET_NAME}', f'arn:aws:s3:::{BUCKET_NAME}/*']},
    {'Effect': 'Allow', 'Action': 'bedrock:InvokeModel', 'Resource': f'arn:aws:bedrock:{REGION}::foundation-model/{EMBEDDING_MODEL}'},
    {'Effect': 'Allow', 'Action': 's3vectors:*', 'Resource': '*'},
    {'Effect': 'Allow', 'Action': 's3:*', 'Resource': 'arn:aws:s3:::*'}
]}
iam.put_role_policy(RoleName=KB_ROLE_NAME, PolicyName='KBPolicy', PolicyDocument=json.dumps(kb_policy))
print('Permissions attached')
time.sleep(10)

---
## Step 5: Create Knowledge Base with S3 Vectors

S3 Vectors provides built-in vector storage at 90% lower cost than OpenSearch Serverless.
First we create an S3 Vector Bucket, then use it as the KB storage backend.

In [ ]:
# Step 5a: Create S3 Vector Bucket and Index
s3vectors = boto3.client('s3vectors', region_name=REGION)
VECTOR_BUCKET_NAME = f'kb-vectors-{ACCOUNT_ID}-{REGION}'
VECTOR_INDEX_NAME = f'kb-index-{time.strftime("%Y%m%d%H%M")}'

# Create vector bucket (idempotent)
try:
    s3vectors.create_vector_bucket(vectorBucketName=VECTOR_BUCKET_NAME)
    print(f'Created vector bucket: {VECTOR_BUCKET_NAME}')
except s3vectors.exceptions.ConflictException:
    print(f'Vector bucket already exists: {VECTOR_BUCKET_NAME}')
except Exception as e:
    if 'Conflict' in str(e) or 'already exists' in str(e).lower():
        print(f'Vector bucket already exists: {VECTOR_BUCKET_NAME}')
    else:
        raise

VECTOR_BUCKET_ARN = f'arn:aws:s3vectors:{REGION}:{ACCOUNT_ID}:bucket/{VECTOR_BUCKET_NAME}'
print(f'Vector Bucket ARN: {VECTOR_BUCKET_ARN}')

# Create vector index (Titan Embed V2 = 1024 dimensions, cosine distance)
try:
    idx_resp = s3vectors.create_index(
        vectorBucketName=VECTOR_BUCKET_NAME,
        indexName=VECTOR_INDEX_NAME,
        dataType='float32',
        dimension=1024,
        distanceMetric='cosine',
        metadataConfiguration={
            'nonFilterableMetadataKeys': ['AMAZON_BEDROCK_TEXT']
        }
    )
    VECTOR_INDEX_ARN = idx_resp['indexArn']
    print(f'Created vector index: {VECTOR_INDEX_NAME}')
except s3vectors.exceptions.ConflictException:
    VECTOR_INDEX_ARN = f'{VECTOR_BUCKET_ARN}/index/{VECTOR_INDEX_NAME}'
    print(f'Vector index already exists: {VECTOR_INDEX_NAME}')
except Exception as e:
    if 'Conflict' in str(e) or 'already exists' in str(e).lower():
        VECTOR_INDEX_ARN = f'{VECTOR_BUCKET_ARN}/index/{VECTOR_INDEX_NAME}'
        print(f'Vector index already exists: {VECTOR_INDEX_NAME}')
    else:
        raise

print(f'Vector Index ARN: {VECTOR_INDEX_ARN}')

# Step 5b: Create Knowledge Base (always new - unique name)
print(f'\nCreating Knowledge Base: {KB_NAME}')
response = bedrock_agent.create_knowledge_base(
    name=KB_NAME,
    description='Amazon SEC filings for cost optimization workshop',
    roleArn=KB_ROLE_ARN,
    knowledgeBaseConfiguration={
        'type': 'VECTOR',
        'vectorKnowledgeBaseConfiguration': {
            'embeddingModelArn': f'arn:aws:bedrock:{REGION}::foundation-model/{EMBEDDING_MODEL}'
        }
    },
    storageConfiguration={
        'type': 'S3_VECTORS',
        's3VectorsConfiguration': {
            'vectorBucketArn': VECTOR_BUCKET_ARN,
            'indexName': VECTOR_INDEX_NAME
        }
    }
)
KB_ID = response['knowledgeBase']['knowledgeBaseId']
print(f'KB created: {KB_ID}')

# Wait for KB to be active
print('Waiting for KB to be ACTIVE...')
for i in range(30):
    status = bedrock_agent.get_knowledge_base(knowledgeBaseId=KB_ID)['knowledgeBase']['status']
    if status == 'ACTIVE':
        print('KB is ACTIVE')
        break
    print('.', end='', flush=True)
    time.sleep(10)

# Save KB_ID to config for notebook 2
config['KB_ID'] = KB_ID
with open('/tmp/certagent_config.json', 'w') as f:
    json.dump(config, f)
print(f'\nKnowledge Base ID: {KB_ID}')

---
## Step 6: Create Data Source and Sync

In [ ]:
# Create data source with fixed-size chunking (required for S3 Vectors metadata limit)
ds_response = bedrock_agent.create_data_source(
    knowledgeBaseId=KB_ID,
    name='sec-filings',
    description='Amazon SEC 10-K filings',
    dataSourceConfiguration={
        'type': 'S3',
        's3Configuration': {
            'bucketArn': f'arn:aws:s3:::{BUCKET_NAME}',
            'inclusionPrefixes': ['documents/']
        }
    },
    vectorIngestionConfiguration={
        'chunkingConfiguration': {
            'chunkingStrategy': 'FIXED_SIZE',
            'fixedSizeChunkingConfiguration': {
                'maxTokens': 300,
                'overlapPercentage': 10
            }
        }
    }
)
DS_ID = ds_response['dataSource']['dataSourceId']
print(f'Data source created: {DS_ID}')

# Start sync
print('Starting data sync...')
sync = bedrock_agent.start_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)
job_id = sync['ingestionJob']['ingestionJobId']

# Wait for sync
for i in range(30):
    job = bedrock_agent.get_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID, ingestionJobId=job_id)['ingestionJob']
    status = job['status']
    if status in ('COMPLETE', 'FAILED'):
        break
    print(f'  [{i}] Sync status: {status}')
    time.sleep(10)

stats = job.get('statistics', {})
print(f'\nSync {status}')
print(f'  Documents scanned: {stats.get("numberOfDocumentsScanned", 0)}')
print(f'  Documents indexed: {stats.get("numberOfNewDocumentsIndexed", 0) + stats.get("numberOfModifiedDocumentsIndexed", 0)}')
if status == 'FAILED':
    print(f'  Failure: {job.get("failureReasons", [])}')

---
## Step 7: Query the Knowledge Base — Baseline

Now query using the Converse API with different model routing strategies.

In [ ]:
def retrieve_and_answer(query, num_results=10, max_tokens=1024, model=None):
    """Retrieve docs from KB and generate answer."""
    model = model or MODEL_ID
    retrieve_resp = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={'text': query},
        retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': num_results}}
    )
    chunks = [r['content']['text'] for r in retrieve_resp.get('retrievalResults', []) if r.get('content', {}).get('text')]
    context = '\n---\n'.join(chunks)
    
    response = bedrock_runtime.converse(
        modelId=model,
        messages=[{'role': 'user', 'content': [{'text': query}]}],
        system=[{'text': f'Answer based on Amazon SEC filings context:\n{context}'}],
        inferenceConfig={'maxTokens': max_tokens}
    )
    usage = response.get('usage', {})
    input_tok = usage.get('inputTokens', 0)
    output_tok = usage.get('outputTokens', 0)
    answer = response['output']['message']['content'][0]['text']
    cost = estimate_cost(model, input_tok, output_tok)
    return {'answer': answer, 'input_tokens': input_tok, 'output_tokens': output_tok, 'cost': cost, 'num_chunks': len(chunks)}

print('retrieve_and_answer() ready')

In [ ]:
# Baseline query
query = 'What was Amazon total revenue in 2023?'
result = retrieve_and_answer(query)
print(f'Model: {MODEL_ID}')
print(f'Chunks: {result["num_chunks"]} | Input: {result["input_tokens"]} | Output: {result["output_tokens"]} | Cost: ${result["cost"]:.6f}')
print()
display(Markdown(result['answer']))

In [ ]:
# Compare Nova Pro vs Nova Lite
query = 'What is AWS revenue and growth rate?'

for model in [MODEL_ID, LITE_MODEL]:
    r = retrieve_and_answer(query, model=model)
    print(f'{model:<35} Cost: ${r["cost"]:.6f} | Tokens: {r["input_tokens"]}+{r["output_tokens"]}')
print('\nNova Lite is 13x cheaper than Nova Pro for the same query.')

---
## Step 8: Create a Prompt Router

A Prompt Router automatically routes requests to the cheapest model that meets quality criteria.
It uses Nova Lite for simple queries and falls back to Nova Pro for complex ones — saving costs automatically.

In [ ]:
# Create a Prompt Router: Nova Lite (cheap) + Nova Pro (fallback)
bedrock_client = boto3.client('bedrock', region_name=REGION)

ROUTER_NAME = 'workshop-cost-router'

# Check if router already exists
routers = bedrock_client.list_prompt_routers()['promptRouterSummaries']
existing_router = next((r for r in routers if r['promptRouterName'] == ROUTER_NAME), None)

if existing_router:
    ROUTER_ARN = existing_router['promptRouterArn']
    print(f'Router already exists: {ROUTER_ARN}')
else:
    router_resp = bedrock_client.create_prompt_router(
        promptRouterName=ROUTER_NAME,
        description='Routes simple queries to Nova Lite and complex ones to Nova Pro',
        models=[
            {'modelArn': f'arn:aws:bedrock:{REGION}::foundation-model/amazon.nova-lite-v1:0'},
            {'modelArn': f'arn:aws:bedrock:{REGION}::foundation-model/amazon.nova-pro-v1:0'},
        ],
        fallbackModel={'modelArn': f'arn:aws:bedrock:{REGION}::foundation-model/amazon.nova-pro-v1:0'},
        routingCriteria={'responseQualityDifference': 25},
    )
    ROUTER_ARN = router_resp['promptRouterArn']
    print(f'Created router: {ROUTER_ARN}')

print(f'\nRouter ARN: {ROUTER_ARN}')

In [ ]:
# Define query_via_router helper
def query_via_router(query):
    """Query KB via Prompt Router and show which model was selected."""
    retrieve_resp = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KB_ID, retrievalQuery={'text': query},
        retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': 5}}
    )
    chunks = [r['content']['text'] for r in retrieve_resp.get('retrievalResults', []) if r.get('content', {}).get('text')]
    context = '\n---\n'.join(chunks)
    response = bedrock_runtime.converse(
        modelId=ROUTER_ARN,
        messages=[{'role': 'user', 'content': [{'text': query}]}],
        system=[{'text': f'Answer from context:\n{context}'}],
        inferenceConfig={'maxTokens': 400}
    )
    trace = response.get('trace', {}).get('promptRouter', {})
    model_used = trace.get('invokedModelId', 'unknown')
    usage = response.get('usage', {})
    cost = estimate_cost(model_used, usage.get('inputTokens', 0), usage.get('outputTokens', 0))
    print(f'Router selected: {model_used}')
    print(f'Cost: ${cost:.6f}')
    print()
    display(Markdown(response['output']['message']['content'][0]['text']))

print('query_via_router() ready')

In [ ]:
# Simple factual query - router should pick Nova Lite (cheaper)
query_via_router('What was Amazon total revenue in 2023?')

In [ ]:
# Complex analytical query - router should pick Nova Pro (smarter)
query_via_router('Compare AWS revenue growth between 2022 and 2023. Explain the key drivers behind the change and analyze whether this trend is sustainable given competitive pressures.')

---
## Step 9: Create an Application Inference Profile (AIP)

AIPs allow you to tag requests for cost tracking and allocation per team/project.
No cost savings on pricing — but enables chargeback and budget control.

In [ ]:
# Create Application Inference Profile
AIP_NAME = 'workshop-nova-pro-aip'

# Check if AIP exists
profiles = bedrock_client.list_inference_profiles(typeEquals='APPLICATION')['inferenceProfileSummaries']
existing_aip = next((p for p in profiles if p['inferenceProfileName'] == AIP_NAME), None)

if existing_aip:
    AIP_ARN = existing_aip['inferenceProfileArn']
    print(f'AIP already exists: {AIP_ARN}')
else:
    aip_resp = bedrock_client.create_inference_profile(
        inferenceProfileName=AIP_NAME,
        description='Nova Pro inference profile for workshop cost tracking',
        modelSource={'copyFrom': f'arn:aws:bedrock:{REGION}::foundation-model/amazon.nova-pro-v1:0'},
        tags=[{'key': 'project', 'value': 'bedrock-workshop'}, {'key': 'team', 'value': 'workshop'}]
    )
    AIP_ARN = aip_resp['inferenceProfileArn']
    print(f'Created AIP: {AIP_ARN}')

print(f'\nAIP ARN: {AIP_ARN}')

In [ ]:
# Query using AIP (same as Nova Pro but with cost tags)
print('=== Query via Application Inference Profile ===')
query = 'What is AWS segment revenue?'

retrieve_resp = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID, retrievalQuery={'text': query},
    retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': 5}}
)
chunks = [r['content']['text'] for r in retrieve_resp.get('retrievalResults', []) if r.get('content', {}).get('text')]
context = '\n---\n'.join(chunks)

response = bedrock_runtime.converse(
    modelId=AIP_ARN,
    messages=[{'role': 'user', 'content': [{'text': query}]}],
    system=[{'text': f'Answer from context:\n{context}'}],
    inferenceConfig={'maxTokens': 300},
    requestMetadata={'team': 'workshop', 'project': 'bedrock-cost-savings'}
)

usage = response.get('usage', {})
cost = estimate_cost(MODEL_ID, usage.get('inputTokens', 0), usage.get('outputTokens', 0))
print(f'AIP model: Nova Pro (tagged for cost allocation)')
print(f'Tokens: {usage} | Cost: ${cost:.6f}')
print(f'Tags: project=bedrock-workshop, team=workshop')
print()
display(Markdown(response['output']['message']['content'][0]['text']))

In [ ]:
# Compare: same query WITHOUT AIP (direct model call)
print('=== Direct Model Call (WITHOUT AIP) ===')
print('Same query, same model, same cost - but NO cost allocation tags.')
print()

response_direct = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[{'role': 'user', 'content': [{'text': query}]}],
    system=[{'text': f'Answer from context:\n{context}'}],
    inferenceConfig={'maxTokens': 300}
)

usage_direct = response_direct.get('usage', {})
cost_direct = estimate_cost(MODEL_ID, usage_direct.get('inputTokens', 0), usage_direct.get('outputTokens', 0))
print(f'Direct call: {MODEL_ID}')
print(f'Tokens: {usage_direct} | Cost: ${cost_direct:.6f}')
print(f'Tags: NONE (no cost attribution)')
print()
display(Markdown(response_direct['output']['message']['content'][0]['text']))
print()
print('=== AIP vs Direct Comparison ===')
print(f'  AIP cost:    ${cost:.6f} (tagged: team=workshop, project=bedrock-workshop)')
print(f'  Direct cost: ${cost_direct:.6f} (no tags - cannot attribute costs)')
print(f'  Price difference: $0 (AIPs are free to create)')
print(f'  Value: Cost visibility, chargeback, and budget control per team/project')

### View Cost Allocation by Tags

With AIPs tagged per team/project, you can query AWS Cost Explorer to see spend breakdown.

In [ ]:
# Query cost allocation by AIP tags using Cost Explorer
ce = boto3.client('ce', region_name='us-east-1')
from datetime import datetime, timedelta

end = datetime.now().strftime('%Y-%m-%d')
start = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')

try:
    cost_resp = ce.get_cost_and_usage(
        TimePeriod={'Start': start, 'End': end},
        Granularity='DAILY',
        Metrics=['UnblendedCost'],
        Filter={
            'Tags': {'Key': 'project', 'Values': ['bedrock-workshop']}
        },
        GroupBy=[{'Type': 'TAG', 'Key': 'team'}]
    )
    print(f'=== Cost by Team (last 7 days, tag: project=bedrock-workshop) ===')
    print(f'Period: {start} to {end}\n')
    total = 0
    for result in cost_resp['ResultsByTime']:
        for group in result.get('Groups', []):
            amount = float(group['Metrics']['UnblendedCost']['Amount'])
            if amount > 0:
                print(f'  {result["TimePeriod"]["Start"]} | {group["Keys"][0]:<20} | ${amount:.4f}')
                total += amount
    if total == 0:
        print('  No tagged costs found yet (tags appear in billing after 24-48 hours)')
    else:
        print(f'\n  Total: ${total:.4f}')
except Exception as e:
    print(f'Cost Explorer query: {e}')
    print('\nNote: Cost allocation tags take 24-48 hours to appear in billing.')
    print('For immediate tracking, use CloudWatch metrics with the AIP ARN.')

print('\nBest practice: Create one AIP per team/project for cost visibility.')

---
## Step 10: Prompt Caching

Prompt caching stores the system prompt (including KB context) so repeated queries don't re-process it.
Saves up to 90% on input tokens for follow-up queries with the same context.

> Requires Claude models (anthropic.claude-sonnet-4-5 or later).

In [ ]:
# Query with Prompt Caching (Claude)
CACHE_MODEL = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

print('=== Query with Prompt Caching ===')
query = 'What was Amazon net income in 2023?'

retrieve_resp = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID, retrievalQuery={'text': query},
    retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': 5}}
)
chunks = [r['content']['text'] for r in retrieve_resp.get('retrievalResults', []) if r.get('content', {}).get('text')]
context = '\n---\n'.join(chunks)

# Baseline call - NO cachePoint (regular pricing)
response0 = bedrock_runtime.converse(
    modelId=CACHE_MODEL,
    messages=[{'role': 'user', 'content': [{'text': query}]}],
    system=[{'text': f'Answer from Amazon SEC filings:\n{context}'}],
    inferenceConfig={'maxTokens': 300}
)
usage0 = response0.get('usage', {})
print(f'Baseline (NO caching):')
print(f'  Input: {usage0.get("inputTokens", 0)} | Output: {usage0.get("outputTokens", 0)}')
print(f'  Cache write: {usage0.get("cacheWriteInputTokens", 0)} | Cache read: {usage0.get("cacheReadInputTokens", 0)}')
print()

# First cached call - cache WRITE (full cost + small write premium)
response1 = bedrock_runtime.converse(
    modelId=CACHE_MODEL,
    messages=[{'role': 'user', 'content': [{'text': query}]}],
    system=[{'text': f'Answer from Amazon SEC filings:\n{context}'}, {'cachePoint': {'type': 'default'}}],
    inferenceConfig={'maxTokens': 300}
)
usage1 = response1.get('usage', {})
print(f'First cached call (cache WRITE):')
print(f'  Input: {usage1.get("inputTokens", 0)} | Output: {usage1.get("outputTokens", 0)}')
print(f'  Cache write tokens: {usage1.get("cacheWriteInputTokens", 0)}')
print()

# Second cached call - same context, different question = cache READ (90% cheaper)
query2 = 'What are Amazon main risk factors?'
response2 = bedrock_runtime.converse(
    modelId=CACHE_MODEL,
    messages=[{'role': 'user', 'content': [{'text': query2}]}],
    system=[{'text': f'Answer from Amazon SEC filings:\n{context}'}, {'cachePoint': {'type': 'default'}}],
    inferenceConfig={'maxTokens': 300}
)
usage2 = response2.get('usage', {})
print(f'Second cached call (cache READ):')
print(f'  Input: {usage2.get("inputTokens", 0)} | Output: {usage2.get("outputTokens", 0)}')
print(f'  Cache read tokens: {usage2.get("cacheReadInputTokens", 0)}')
print()

# Compare all three
print('=== Cost Comparison ===')
baseline_input = usage0.get('inputTokens', 0)
cached_read = usage2.get('cacheReadInputTokens', 0)
print(f'  Baseline (no cache):  {baseline_input} input tokens at full price')
print(f'  Cache write call:     {usage1.get("cacheWriteInputTokens", 0)} tokens written (25% premium on write)')
print(f'  Cache read call:      {cached_read} tokens read at 90% discount')
print(f'\nCache read tokens are 90% cheaper than regular input tokens!')
print()
display(Markdown(response2['output']['message']['content'][0]['text']))

---
## Step 13: Optimization 1 — Context Window Reduction

Retrieving 10 chunks when 3 are enough wastes input tokens.

In [ ]:
query = 'What was Amazon operating income in 2023?'

print('=== Context Window Comparison ===')
for n in [3, 5, 10]:
    r = retrieve_and_answer(query, num_results=n)
    print(f'  Chunks: {r["num_chunks"]:>2} | Input: {r["input_tokens"]:>5} | Cost: ${r["cost"]:.6f}')
    if n == 3:
        print(f'  Sample answer (3 chunks): {r["answer"][:200]}')
        print()
print('\nReduction from 10 to 3 chunks saves 40-60% on input costs.')

---
## Step 14: Optimization 2 — Output Token Limits

Cap maxTokens based on query type.

In [ ]:
query = 'What is Amazon employee count?'

print('=== Output Token Limit Comparison ===')
for max_tok in [100, 200, 500, 1024]:
    r = retrieve_and_answer(query, num_results=3, max_tokens=max_tok)
    print(f'  maxTokens: {max_tok:>4} | Actual: {r["output_tokens"]:>4} | Cost: ${r["cost"]:.6f}')
    if max_tok == 100:
        display(Markdown(f'**Answer at 100 tokens:** {r["answer"][:300]}'))
        print()
print('\nShort factual answers need only 100-200 tokens.')

---
## Step 15: Optimization 3 — Streaming with Early Stop

Stop generation when a complete answer is detected.

In [ ]:
def stream_early_stop(query, max_chars=400):
    stop_phrases = ['\n\nIs there anything', '\n\nLet me know', '\n\nWould you like', '\n\nPlease note']
    retrieve_resp = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KB_ID, retrievalQuery={'text': query},
        retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': 3}}
    )
    chunks = [r['content']['text'] for r in retrieve_resp.get('retrievalResults', []) if r.get('content', {}).get('text')]
    context = '\n---\n'.join(chunks)
    
    response = bedrock_runtime.converse_stream(
        modelId=MODEL_ID,
        messages=[{'role': 'user', 'content': [{'text': query}]}],
        system=[{'text': f'Answer concisely:\n{context}'}],
        inferenceConfig={'maxTokens': 1024}
    )
    text = ''
    stopped = False
    for event in response['stream']:
        if 'contentBlockDelta' in event:
            text += event['contentBlockDelta']['delta'].get('text', '')
            if len(text) > max_chars:
                stopped = True; break
            for p in stop_phrases:
                if p in text:
                    text = text[:text.index(p)]; stopped = True; break
            if stopped: break
    return {'answer': text.strip(), 'stopped_early': stopped, 'chars': len(text)}

query = 'What are Amazon main business segments?'
full = retrieve_and_answer(query, num_results=3, max_tokens=1024)
streamed = stream_early_stop(query)
print(f'Full:       {len(full["answer"]):>4} chars, cost=${full["cost"]:.6f}')
print(f'Early stop: {streamed["chars"]:>4} chars, stopped={streamed["stopped_early"]}')
print(f'Savings: {100 - (streamed["chars"]*100//len(full["answer"]))}% fewer output chars')
print()
display(Markdown(f'**Streamed answer (early stop):** {streamed["answer"][:300]}'))

---
## Step 16: Optimization 4 — Batch Inference (50% off)

For non-real-time workloads, batch inference costs half.

In [ ]:
batch_queries = [
    'What was Amazon total revenue?',
    'What is AWS revenue?',
    'How many employees does Amazon have?',
    'What was Amazon net income?',
    'What are the main risk factors?',
]

# Prepare batch records
batch_records = []
for i, q in enumerate(batch_queries):
    resp = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KB_ID, retrievalQuery={'text': q},
        retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': 3}}
    )
    chunks = [r['content']['text'] for r in resp.get('retrievalResults', [])[:3] if r.get('content', {}).get('text')]
    batch_records.append({'recordId': f'q-{i}', 'modelInput': {
        'messages': [{'role': 'user', 'content': [{'text': q}]}],
        'system': [{'text': f'Answer from context:\n{chr(10).join(chunks)}'}],
        'inferenceConfig': {'maxTokens': 200}
    }})

print(f'Batch records prepared: {len(batch_records)}')
print('In production: upload JSONL to S3, call create_model_invocation_job()')
print()

# Show what a batch answer looks like (simulated via on-demand)
sample = retrieve_and_answer(batch_queries[0], num_results=3, max_tokens=200)
display(Markdown(f'**Sample batch answer** ({batch_queries[0]}): {sample["answer"][:300]}'))
print()

# Cost estimate
avg = sum(retrieve_and_answer(q, num_results=3, max_tokens=200)['cost'] for q in batch_queries[:2]) / 2
print(f'On-demand (5 queries): ${avg*5:.6f}')
print(f'Batch (50% off):       ${avg*5*0.5:.6f}')
print(f'At 1000/day savings:   ${avg*500*30:.2f}/month')

---
## Step 17: Optimization 5 — Model Distillation

Train Nova Lite on your specific Q&A pairs for 92% cost reduction.

In [ ]:
print('Note: This step demonstrates the workflow conceptually. Actual model fine-tuning')
print('requires a separate training job that takes hours to complete.')
print()
print('=== Model Distillation Workflow ===')
print()
print('1. Collect training data: log queries + Nova Pro responses')
print('2. Fine-tune Nova Lite on 500-1000 Q&A pairs (~$5-20)')
print('3. Deploy distilled model \u2014 same quality, 92% cheaper')
print()
print(f'Nova Pro:  ${PRICING[MODEL_ID]["input"]}/1K input')
print(f'Nova Lite: ${PRICING[LITE_MODEL]["input"]}/1K input')
print(f'Savings:   {100 - int(PRICING[LITE_MODEL]["input"]/PRICING[MODEL_ID]["input"]*100)}%')
print()
print('For this workshop KB, Nova Lite already works well for simple factual queries:')
r = retrieve_and_answer('What was Amazon 2023 revenue?', num_results=3, max_tokens=150, model=LITE_MODEL)
print(f'  Nova Lite answer: {r["answer"][:200]}')
print(f'  Cost: ${r["cost"]:.6f}')

---
## Summary: Combined Savings

In [ ]:
print('=== Cost Optimization Summary ===')
print()
print(f'{"Technique":<35} {"Savings":<12} {"Effort"}')
print('-' * 60)
for name, sav, eff in [
    ('Context reduction (10->3)', '40-60%', 'Easy'),
    ('Output token limits', '20-70%', 'Easy'),
    ('Streaming early stop', '10-30%', 'Easy'),
    ('Batch inference', '50%', 'Medium'),
    ('Model distillation', '85-93%', 'High'),
]: print(f'{name:<35} {sav:<12} {eff}')
print()
print('Apply in order: 1 -> 2 -> 3 -> 4 -> 5')

---
## Cleanup

Delete the Knowledge Base and associated resources.

In [ ]:
# Uncomment to clean up:
# bedrock_agent.delete_data_source(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)
# bedrock_agent.delete_knowledge_base(knowledgeBaseId=KB_ID)
# s3_resource = boto3.resource('s3')
# bucket = s3_resource.Bucket(BUCKET_NAME)
# bucket.objects.all().delete()
# bucket.delete()
# iam.delete_role_policy(RoleName=KB_ROLE_NAME, PolicyName='KBPolicy')
# iam.delete_role(RoleName=KB_ROLE_NAME)
# print('All resources deleted')

print('Uncomment the lines above and run to delete all workshop resources.')